<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex04-perceptron-to-mlp/Ex04_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_04 · Notebook 00 — Environment Check

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook is **read only**. Run it from top to bottom and check each output
against the note underneath it. It takes about a minute.

## What Ex_04 is about

L4.1 took you from a straight line to one neuron to a network. L4.2 took you
from "what can this represent" to "what should it represent". Ex_04 is those two
lectures done with your own hands, in six notebooks:

| | | |
|---|---|---|
| 01 | perceptron and XOR, in NumPy | one neuron, its learning rule, and the failure that stopped the field for fifteen years |
| 02 | the same network in PyTorch | how much of notebook 01 was bookkeeping |
| 03 | counting kinks | a shallow ReLU network is piecewise linear, with one kink per unit |
| 04 | depth against width | one parameter budget, spent three ways, measured |
| 05 | overfit, then regularise | **the eleven points from L4.2 slides 11-13**, on your own screen |
| 06 | the report | the four questions, and a model you have to defend |

Notebook 01 is meant to fail partway through. It is the only notebook in this
course that is, and it says so loudly in three places, because every year
somebody reports it as a bug.

## How to run these notebooks

- **Google Colab**: upload the whole `Ex04-perceptron-to-mlp` folder. Everything
  needed is preinstalled. No GPU — the largest network here has about a thousand
  parameters.
- **Locally**: any Python 3.9 or newer with `torch`, `numpy` and `matplotlib`.
- Run them **in order**. Notebook 06 reads results written by 03, 04 and 05.
- Keep `Ex_4_core.py` next to the notebooks.

---

## 1 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_4_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex04-perceptron-to-mlp/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import platform

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch

print("python      ", sys.version.split()[0], "on", platform.system())
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)
print("torch       ", torch.__version__)

**What you should see.** Four version strings and no traceback. Python 3.9 or
newer and torch 2.x. Nothing in Ex_04 uses a recent feature of anything.

---

## 2 · The shared module

`Ex_4_core.py` holds three datasets, a configurable network, a training loop and
the plotting helpers. It is complete; your work is in the `# TODO:` cells.

In [ ]:
import Ex_4_core as core

print("module loaded from:", core.__file__)
print()
X, y = core.logic_dataset("XOR")
print("XOR inputs:\n", X)
print("XOR targets:", y)

**What you should see.** The path to the module, a 4 × 2 array of the four input
rows in the order (0,0), (0,1), (1,0), (1,1), and the targets `[0, 1, 1, 0]`.

Four rows. That is the entire dataset that held the field up for fifteen years.

---

## 3 · The lecture's eleven points

Notebook 05 uses the dataset from **L4.2 slides 11 to 13** — the eleven noisy
samples that the lecture fitted three ways: too rigid, about right, too flexible.

These are not "similar" points regenerated with a random seed. They are the same
eleven numbers, copied out of the lecture's own generator, so that the figure you
produce in notebook 05 is the figure that was on the screen in the lecture
theatre.

In [ ]:
x11, y11 = core.lecture_dataset()

print(" i     x       y")
for i in range(11):
    print(f"{i:2d}   {x11[i]:.1f}   {y11[i]:+.4f}")
print()
print("noise values baked into the lecture:", core.LECTURE_NOISE)

In [ ]:
x_val, y_val = core.lecture_validation()
xs = np.linspace(0, 1, 400)

fig, ax = plt.subplots(figsize=(7.4, 4.4))
core.plot_fit(x11, y11, truth=core.lecture_truth, x_val=x_val, y_val=y_val,
              x_curve=xs, ax=ax,
              title="L4.2 slides 11-13: eleven samples, and a held-out set")
plt.show()

**What you should see.** Eleven black circles that rise from about 0.36 to a
plateau near 0.9 and come back down to 0.46, a grey dashed curve through the
middle of them, and forty green squares scattered around the same curve.

The dashed curve is the truth, `0.30 + 0.52 sin(2.9x) + 0.10x`. The lecture is
careful to say that in real life you never see it, and that remains true here:
notebook 05 uses it only to score the models at the end.

The green squares are a **held-out set** — a second measurement campaign on the
same process with the same instrument. Notebook 05 is entirely about the
difference between what the black circles tell you and what the green squares
tell you.

---

## 4 · The regression target for notebooks 03 and 04

A different curve, with two length scales, so that a network needs several kinks
before it can follow it.

In [ ]:
xw, yw = core.wiggly_dataset(n=400, noise=0.06)
xg = np.linspace(-1, 1, 400)

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.plot(xg, core.wiggly_truth(xg), lw=1.6, ls="--", color="#888888",
        label="truth")
ax.plot(xw, yw, ".", ms=3, color="#111111", label="400 samples")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("The capacity target:  sin(πx) + 0.35 sin(3πx)  on [-1, 1]")
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.25)
plt.show()

print("samples:", xw.shape[0], " domain:", core.WIGGLY_DOMAIN, " noise sd: 0.06")

**What you should see.** One large oscillation with a smaller ripple riding on top
of it, running from $x = -1$ to $x = +1$, and 400 closely spaced samples around
it. Notebooks 03 and 04 both use this target; notebook 05 uses the lecture's
eleven points instead.

---

## 5 · PyTorch trains

The last check. Build a small network, train it briefly on the wiggly data with
a held-out set, and plot both curves on the same axes.

**Both curves on the same axes is a standing requirement of this course.** Every
training run you report in Ex_04 must be shown this way. A training curve on its
own hides the only thing you are looking for.

In [ ]:
core.set_seed(0)

x_tr, y_tr = core.wiggly_dataset(n=200, noise=0.06, seed=7)
x_va, y_va = core.wiggly_dataset(n=200, noise=0.06, seed=8)

model = core.MLP(hidden=(16,), activation="relu")
history = core.train(model, x_tr, y_tr, x_val=x_va, y_val=y_va,
                     epochs=600, lr=2e-2)

print("parameters:", core.count_parameters(model))
print(f"train loss  first {history['train'][0]:.4f}  last {history['train'][-1]:.4f}")
print(f"val   loss  first {history['val'][0]:.4f}  last {history['val'][-1]:.4f}")

core.plot_curves(history, title="Environment check — 16 hidden units, 600 epochs")
plt.show()

**What you should see.** `49` parameters, both losses falling from around 0.5 to
somewhere below 0.05, and two lines on one log-scaled plot that stay close
together for the whole run.

The absolute numbers depend on your PyTorch version. The two curves lying on top
of one another is the part that matters.

Close together is the healthy picture: a model with 49 parameters fitted to 200
points has no room to memorise anything. In notebook 05 you will see the
unhealthy picture, deliberately.

If the loss prints `nan`, or does not fall at all, fix the installation before
going further.

---

## 6 · Ready

Continue with **`Ex04_01_perceptron_and_xor.ipynb`**.

One warning to read before you start it. **Notebook 01 contains a training loop
that is supposed to fail.** It will run, print sensible-looking output, and never
converge, and that is the correct result. The notebook says so three times. Do
not spend an hour debugging it.